# 02 — Forensic Tools & Local-Model Reliability

The #1 killer of local-LLM agent projects is **tool-calling flakiness**: a 8-9B
quantized model that forgets the format, invents arguments, or rambles instead of
answering. This notebook shows the two layers of defense this project uses, with
measurements, not folklore:

1. **Tool design** — deterministic tools do the analytical heavy lifting (SQL,
   fuzzy matching); the model only *picks* tools and reads compact summaries.
   Full results go to an **evidence store**, not into the context window.
2. **A measured reliability ladder for structured output** — we benchmark two
   extraction methods on both local backbones and discover a genuine Ollama
   gotcha along the way.

Everything below runs against the warehouse from notebook 01.

In [1]:
import json
import time

import httpx
import pandas as pd
from pydantic import ValidationError

from aml_investigator import db
from aml_investigator.settings import settings
from aml_investigator.tools.forensics import make_tools, set_active_case

pd.set_option("display.max_colwidth", 160)

con = db.connect()                                          # open the warehouse from nb01
con.execute("DELETE FROM evidence WHERE case_id = 'nb02-demo'")  # clean slate for this demo
set_active_case("nb02-demo")    # tools tag every evidence row with the active case id
tools = make_tools(con)         # build the toolset bound to this connection
list(tools)

['profile_account',
 'velocity_scan',
 'structuring_scan',
 'counterparty_network',
 'sanctions_check',
 'run_sql']

## The forensic toolset

Five specialised scans plus one guarded ad-hoc SQL tool. Note the design contract
visible in every output below: results come back as `[EV-xx] <compact summary>`.
The full payload (every row, every match) is persisted to the `evidence` table
under that id — the model's context stays small, and later the report validator
can check every claimed number against what the tools *actually* returned.

### `structuring_scan` — on a known structuring account

In [2]:
struct_acc = con.execute(
    "SELECT account_id FROM ground_truth WHERE typology = 'structuring' LIMIT 1").fetchone()[0]
print(tools["structuring_scan"].invoke({"account_id": struct_acc}))

[EV-01] ACC-0046: 12 cash deposits in the $8,500-$10,000 band totalling $112,447 between 2026-04-16 and 2026-04-21 (max 2/day); 0 deposits >= $10,000 ($0).


### `velocity_scan` — dormant account, 3-day burst

In [3]:
vel_acc = con.execute(
    "SELECT account_id FROM ground_truth WHERE typology = 'velocity_burst' LIMIT 1").fetchone()[0]
print(tools["velocity_scan"].invoke({"account_id": vel_acc}))

[EV-02] ACC-0053: median 0 txns/day (full calendar); busiest 3-day window starts 2026-05-18 with 36 txns / $185,899 — burst ratio 24x baseline. Top days: [('2026-05-18', 12, 56469), ('2026-05-19', 12, 59937), ('2026-05-20', 12, 69494)]


### `counterparty_network` — ring detection via recursive SQL

In [4]:
ring_acc = con.execute(
    "SELECT account_id FROM ground_truth WHERE typology = 'circular_transfers' LIMIT 1").fetchone()[0]
print(tools["counterparty_network"].invoke({"account_id": ring_acc}))

[EV-03] ACC-0061: 3 distinct senders, 142 distinct receivers. Top counterparties: out Nicholas Mcbride (US) $89,974; in Hudson, Adams and Wheeler (US) $5,053; in Larson, Page and Norman (US) $5,053; in Buckley-Shepherd (US) $5,053. CIRCULAR RING(S) DETECTED: [{'ring': ['ACC-0061', 'ACC-0062', 'ACC-0063'], 'total_transferred': 271240.0}, {'ring': ['ACC-0061', 'ACC-0117', 'ACC-0135'], 'total_transferred': 2477.0}]


Note the output contains both the **planted ring** (~$270k) *and* a tiny organic
"ring" from random baseline transfers (a few thousand dollars). Real forensic
tools produce noise; deciding which cycle matters is exactly the risk analyst's
job in notebook 03.

### `sanctions_check` — fuzzy screening against the real OFAC list

In [5]:
sanc_acc = con.execute(
    "SELECT account_id FROM ground_truth WHERE typology = 'sanctioned_counterparty' LIMIT 1").fetchone()[0]
print(tools["sanctions_check"].invoke({"account_id": sanc_acc}))

[EV-04] ACC-0081: screened 138 counterparty names against 19,065 OFAC SDN entries (fuzzy cutoff 87). 2 HIT(S): 'Islamic Armu Of Aden' ~ 'ISLAMIC ARMY OF ADEN' (score 95.0, SDGT); 'Haqqaui, Nasiruddin' ~ 'HAQQANI, Nasiruddin' (score 94.4, SDGT)


In [6]:
# ... and on a busy clean account, to see the false-positive behaviour of fuzzy matching
print(tools["sanctions_check"].invoke({"account_id": "ACC-0011"})
      if sanc_acc != "ACC-0011" else tools["sanctions_check"].invoke({"account_id": "ACC-0001"}))

[EV-05] ACC-0011: screened 253 counterparty names against 19,065 OFAC SDN entries (fuzzy cutoff 87). 3 HIT(S): 'Valfajr Shipping eompany Pjs' ~ 'VALFAJR SHIPPING COMPANY PJS' (score 96.4, IRAN); 'Kamyshev, Denss Valentinovich' ~ 'KAMYSHEV, Denis Valentinovich' (score 96.4, RUSSIA-EO14024); 'Ramos Group' ~ 'RAMOR GROUP' (score 90.9, NPWMD] [IFSR)


Fuzzy screening at cutoff 87 catches one-character evasions (scores 94-96) but can
also surface coincidental near-matches of legitimate names. That trade-off
(cutoff vs false positives) is a **policy decision**, which is why the score and
the program land in the evidence rather than a binary verdict.

### `run_sql` — guarded ad-hoc analysis

The agent gets real SQL power, fenced three ways: SELECT-only (parsed with
`sqlglot`, not regex), table allowlist, auto-`LIMIT`. The allowlist is also the
**anti-cheating mechanism**: `ground_truth` (the eval labels!) is invisible.

In [7]:
print(tools["run_sql"].invoke({"query":
    "SELECT counterparty_country, count(*) n, round(sum(amount)) total "
    f"FROM transactions WHERE account_id = '{vel_acc}' AND direction = 'out' "
    "GROUP BY 1 ORDER BY total DESC"}))

[EV-06] run_sql returned 5 row(s) for: SELECT counterparty_country, count(*) n, round(sum(amount)) total FROM transactions WHERE account_id = 'ACC-0053' AND di
  counterparty_country  n    total
0                   SY  9  46757.0
1                   MM  4  17997.0
2                   PA  3  17988.0
3                   IR  2   8782.0
4                   KP  1   6998.0


In [8]:
print(tools["run_sql"].invoke({"query": "DROP TABLE transactions"}))
print(tools["run_sql"].invoke({"query": "SELECT * FROM ground_truth"}))

ERROR: Query rejected: only SELECT queries are allowed.
ERROR: Query rejected: table 'ground_truth' is not accessible. Available tables: accounts, alerts, sdn, transactions.


## The evidence store (state compression in action)

Every tool call above persisted its full payload. The model only ever saw the
short summaries — here's the size difference that keeps an 8B model's context
manageable across a multi-step investigation:

In [9]:
con.execute("""
    SELECT evidence_id, tool, length(summary) AS summary_chars, length(payload) AS payload_chars
    FROM evidence WHERE case_id = 'nb02-demo' ORDER BY evidence_id""").fetch_df()

,evidence_id,tool,summary_chars,payload_chars
0,EV-01,structuring_scan,147,207
1,EV-02,velocity_scan,227,229
2,EV-03,counterparty_network,407,1495
3,EV-04,sanctions_check,235,323
4,EV-05,sanctions_check,339,495
5,EV-06,run_sql,151,477


## Structured output on local models: measure, don't assume

Agent graphs need *validated* structured decisions (triage priorities, risk
scores). There are two mechanisms on Ollama:

- **`function_calling`** — the model emits a tool call; arguments are parsed.
- **`json_schema`** — Ollama constrains decoding with a grammar built from the
  JSON schema (the model physically cannot emit invalid JSON).

### First, a genuine gotcha we hit while building this

On Ollama 0.30.x with a thinking model (qwen3.5), **setting `think: false`
silently disables `format=` constrained decoding** — you get free-form markdown
back. With thinking left on, the thinking goes to a separate field and the
content is perfectly constrained. Demonstrated live:

In [10]:
schema = {"type": "object",
          "properties": {"priority": {"type": "string", "enum": ["high", "medium", "low"]},
                          "rationale": {"type": "string"}},
          "required": ["priority", "rationale"]}
msg = [{"role": "user", "content":
        "Priority for: 14 deposits of $9,900 in 3 days then wired abroad?"}]

for think in (False, None):
    body = {"model": "qwen3.5:9b", "stream": False, "format": schema,
            "options": {"temperature": 0}, "messages": msg}
    if think is not None:
        body["think"] = think
    r = httpx.post(f"{settings.ollama_base_url}/api/chat", json=body, timeout=300).json()
    content = r["message"]["content"]
    try:
        json.loads(content)
        verdict = "VALID JSON"
    except json.JSONDecodeError:
        verdict = "NOT JSON (constrained decoding silently disabled!)"
    print(f"think={think}: {verdict}\n  content head: {content[:110]!r}\n")

think=False: NOT JSON (constrained decoding silently disabled!)
  content head: 'Based on standard banking regulations (such as the U.S. Bank Secrecy Act and Anti-Money Laundering guidelines)'



think=None: VALID JSON
  content head: '{ "priority": "high", "rationale": "This scenario exhibits multiple classic indicators of money laundering and'



### The benchmark: 2 backbones x 2 methods x 3 triage prompts

Small but real: every cell below actually calls the local models. We measure
**validity** (parsed into the Pydantic schema without error) and **latency**.

In [11]:
from aml_investigator.llm import get_chat_model
from aml_investigator.schemas import TriageDecision

PROMPTS = [
    "Alert: account received 14 cash deposits of $9,900 each over 3 days (just under "
    "the $10,000 reporting threshold), then wired the full balance abroad.",
    "Alert: dormant personal account suddenly moved $180,000 in and out within 72 hours "
    "via 30 transfers, destinations include high-risk countries.",
    "Alert: periodic KYC review of a business account with frequent large wires to "
    "counterparties in AE and TR; one name resembles an OFAC-listed entity.",
]
SYSTEM = ("You are an AML triage officer. Select priority and forensic checks "
          "(each at most once) from: profile_account, structuring_scan, velocity_scan, "
          "counterparty_network, sanctions_check.")

rows = []
for model in ("granite4.1:8b", "qwen3.5:9b"):
    for method in ("function_calling", "json_schema"):
        for i, prompt in enumerate(PROMPTS):
            llm = get_chat_model(model).with_structured_output(TriageDecision, method=method)
            t0 = time.perf_counter()
            try:
                result = llm.invoke([("system", SYSTEM), ("user", prompt)])
                ok, n_checks = result is not None, len(result.checks) if result else 0
            except Exception as e:
                ok, n_checks = False, 0
            rows.append({"model": model, "method": method, "prompt": i,
                         "valid": ok, "seconds": round(time.perf_counter() - t0, 1),
                         "n_checks": n_checks})
            print(rows[-1])

bench = pd.DataFrame(rows)
out_dir = settings.artifacts_dir / "reliability"
out_dir.mkdir(parents=True, exist_ok=True)
bench.to_csv(out_dir / "structured_output_benchmark.csv", index=False)

{'model': 'granite4.1:8b', 'method': 'function_calling', 'prompt': 0, 'valid': True, 'seconds': 6.9, 'n_checks': 3}


{'model': 'granite4.1:8b', 'method': 'function_calling', 'prompt': 1, 'valid': True, 'seconds': 3.1, 'n_checks': 3}


{'model': 'granite4.1:8b', 'method': 'function_calling', 'prompt': 2, 'valid': True, 'seconds': 2.3, 'n_checks': 2}


{'model': 'granite4.1:8b', 'method': 'json_schema', 'prompt': 0, 'valid': True, 'seconds': 2.0, 'n_checks': 3}


{'model': 'granite4.1:8b', 'method': 'json_schema', 'prompt': 1, 'valid': True, 'seconds': 2.5, 'n_checks': 5}


...

{'model': 'qwen3.5:9b', 'method': 'function_calling', 'prompt': 1, 'valid': True, 'seconds': 15.2, 'n_checks': 4}


{'model': 'qwen3.5:9b', 'method': 'function_calling', 'prompt': 2, 'valid': True, 'seconds': 14.2, 'n_checks': 3}


{'model': 'qwen3.5:9b', 'method': 'json_schema', 'prompt': 0, 'valid': True, 'seconds': 135.6, 'n_checks': 2}


{'model': 'qwen3.5:9b', 'method': 'json_schema', 'prompt': 1, 'valid': False, 'seconds': 201.5, 'n_checks': 0}


{'model': 'qwen3.5:9b', 'method': 'json_schema', 'prompt': 2, 'valid': True, 'seconds': 72.8, 'n_checks': 3}


In [12]:
summary = bench.groupby(["model", "method"]).agg(
    validity_rate=("valid", "mean"),
    median_seconds=("seconds", "median"),
    mean_checks=("n_checks", "mean"),
).round(2)
summary

validity_rate  median_seconds  mean_checks
model         method                                                      
granite4.1:8b function_calling           1.00             3.1         2.67
              json_schema                1.00             2.1         3.33
qwen3.5:9b    function_calling           1.00            15.2         3.33
              json_schema                0.67           135.6         1.67

### What the numbers mean (and the design they led to)

- **granite4.1:8b + function_calling** is the sweet spot on this hardware —
  single-digit seconds with full validity. That's why it's the default backbone.
- **qwen3.5:9b pays a thinking tax** on every structured call; with `json_schema`
  (which requires thinking left on, per the gotcha above) it can take minutes.
- `json_schema` is the *reliability* backstop: grammar-constrained decoding cannot
  produce unparseable output, even when it's slow.

Hence the project's `structured_llm_call` escalation ladder, used by every
graph node in notebook 03:

```
function_calling  →  json_schema (+ error feedback)  →  deterministic fallback
   (fast path)          (guaranteed parseable)            (graph never dies)
```

In [13]:
from aml_investigator.llm import structured_llm_call

outcome = structured_llm_call(
    TriageDecision, SYSTEM, PROMPTS[0], name="nb02-demo",
    fallback=TriageDecision(priority="high", checks=["profile_account", "sanctions_check"],
                            rationale="fallback"))
print(f"method={outcome.method_used} attempts={outcome.attempts} {outcome.seconds}s")
outcome.value.model_dump()

method=function_calling attempts=1 6.17s


{'priority': 'high',
 'checks': ['structuring_scan', 'velocity_scan', 'sanctions_check'],
 'rationale': 'The pattern of 14 deposits just under $10,000 each within three days is a classic structuring technique to evade reporting thresholds. The rapid movement of the full balance abroad further suggests potential money laundering or illicit financing. Therefore, high priority with structuring and velocity scans plus a sanctions check are warranted.'}

In [14]:
con.close()